# Data Ingestion

This notebook shows the data ingestion step including checking the sanity of the table in the database.

# Append Project Path

In [1]:
from env_path import append_env_path
append_env_path()

# Test Data Connection

In [2]:
from classification.db_connection import get_mariadb_connection, get_sqlalchemy_connection

In [3]:
conn = get_mariadb_connection()
cur = conn.cursor()

cur.execute("SHOW TABLES;")
tables = cur.fetchall()

cur.close()
conn.close()

In [4]:
print(tables)

[('account',), ('card',), ('client',), ('disp',), ('district',), ('loan',), ('order',), ('trans',)]


There are 8 tables in the database. This aligns with the data description given by the source. <br>
![Here](https://relational.fel.cvut.cz/assets/img/datasets-generated/financial.svg)

# Data Sanity Check

Quick look at the overall data before diving into EDA. What are the missing data like? What each column means? etc.

The data dictionary can be found [here](https://web.archive.org/web/20180506035658/http://lisp.vse.cz/pkdd99/Challenge/berka.htm).

In [22]:
import pandas as pd

In [43]:
conn = get_sqlalchemy_connection() # Use SQLAlchemy to keep pandas happy 🙃.

In [24]:
def query(qry) -> pd.DataFrame:
    """Queries the financial database and returns a pandas dataframe."""
    result = pd.read_sql(qry, con=conn)
    return result

## Account

In [25]:
query("SELECT * FROM account LIMIT 10;")

,account_id,district_id,frequency,date
0,1,18,POPLATEK MESICNE,1995-03-24
1,2,1,POPLATEK MESICNE,1993-02-26
2,3,5,POPLATEK MESICNE,1997-07-07
3,4,12,POPLATEK MESICNE,1996-02-21
4,5,15,POPLATEK MESICNE,1997-05-30
5,6,51,POPLATEK MESICNE,1994-09-27
6,7,60,POPLATEK MESICNE,1996-11-24
7,8,57,POPLATEK MESICNE,1995-09-21
8,9,70,POPLATEK MESICNE,1993-01-27
9,10,54,POPLATEK MESICNE,1996-08-28


In [26]:
query("SELECT COUNT(DISTINCT account_id) FROM account;")

,COUNT(DISTINCT account_id)
0,4500


In [27]:
query("SELECT COUNT(DISTINCT district_id) FROM account;")

,COUNT(DISTINCT district_id)
0,77


The data consists of 4,500 accounts from 77 branches.

In [28]:
query("SELECT DISTINCT frequency FROM account;")

,frequency
0,POPLATEK MESICNE
1,POPLATEK TYDNE
2,POPLATEK PO OBRATU


The frequency are in Czech. This can be translated or encoded to improve readability for English speakers.

In [29]:
query("SELECT MIN(date), MAX(date) FROM account;")

,MIN(date),MAX(date)
0,1993-01-01,1997-12-29


The account creation date spanned from start of 1993 to end of 1997.

## Client

In [30]:
query("SELECT * FROM client LIMIT 10;")

,client_id,gender,birth_date,district_id
0,1,F,1970-12-13,18
1,2,M,1945-02-04,1
2,3,F,1940-10-09,1
3,4,M,1956-12-01,5
4,5,F,1960-07-03,5
5,6,M,1919-09-22,12
6,7,M,1929-01-25,15
7,8,F,1938-02-21,51
8,9,M,1935-10-16,60
9,10,M,1943-05-01,57


In [31]:
query("SELECT COUNT(DISTINCT client_id) FROM client;")

,COUNT(DISTINCT client_id)
0,5369


In [45]:
query("""
SELECT
    gender,
    COUNT(DISTINCT client_id) AS 'count',
    COUNT(DISTINCT client_id) * 100.0 / (SELECT COUNT(*) FROM client) AS 'pct'
FROM client
GROUP BY gender;
""")

,gender,count,pct
0,F,2645,49.2643
1,M,2724,50.7357


There are 5,369 client ids and about half of the clients are females and the other half are males.

In [46]:
query("""
SELECT
    MAX(birth_date),
    MIN(birth_date)
FROM
    client;
""")

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)